# SmolVLM2 Vision Eval (Kaggle T4)

Val-select the 5 Hub checkpoints (60/120/180/240/300) on frozen val-40, then test the winner once on clean-100. Same EXTRACT_PROMPT constant, same metrics.py scorer, same native-resolution tiling as training. Results sync to Hub + kernel output.

In [ ]:
import subprocess, sys, os, time, json
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
REPO_URL = 'https://github.com/eulogik/TinyDoc-VLM'
REPO = '/kaggle/working/tinydoc-vlm'
from huggingface_hub import snapshot_download as _snapdl
import os as _os2
EVAL_DS = _snapdl('eulogik/TinyDoc-VLM-clean-eval', repo_type='dataset',
                 token=_os2.environ.get('HF_TOKEN'))
HUB_AD = os.environ.get('HUB_AD', 'eulogik/SmolVLM2-TinyDoc-distill-vision')
HUB_RES = os.environ.get('HUB_RES', 'eulogik/SmolVLM2-TinyDoc-distill-vision')
CKPTS = os.environ.get('CHECKPOINTS', '60 120 180 240 300').split()
if not os.environ.get('HF_TOKEN'):
    try:
        from kaggle_secrets import UserSecretsClient
        _c = UserSecretsClient()
        for _ in range(5):
            try:
                os.environ['HF_TOKEN'] = _c.get_secret('HF_TOKEN')
                break
            except Exception:
                time.sleep(5)
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    print('WARNING: no HF_TOKEN secret - Hub sync disabled, results via kernel output only')
if os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO])
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.50', 'trl>=0.14', 'peft', 'bitsandbytes',
    'pillow', 'datasets', 'huggingface_hub', 'num2words'])
VAL = f'{EVAL_DS}/manifest_val.json'
TEST = f'{EVAL_DS}/manifest_test.json'
BASE = 'HuggingFaceTB/SmolVLM2-2.2B-Instruct'
def run(adapter, manifest, suffix, limit=None):
    cmd = [sys.executable, f'{REPO}/evaluation/phase0/run_torch_adapter_eval.py',
           '--model', BASE, '--eval-path', manifest, '--out-suffix', suffix,
           '--max-tokens', '256']
    if adapter:
        cmd += ['--adapter', adapter]
    if limit:
        cmd += ['--limit', str(limit)]
    p = subprocess.run(cmd, cwd=EVAL_DS)
    p.check_returncode()
def get_adapter(ckpt):
    from huggingface_hub import snapshot_download
    import os as _os
    snap = snapshot_download(f'{HUB_AD}', allow_patterns=[f'{ckpt}/*'],
                              token=_os.environ.get('HF_TOKEN'))
    return os.path.join(snap, ckpt)
results = f'{REPO}/evaluation/phase0/results'
for ck in CKPTS:
    ad = get_adapter(f'checkpoint-{ck}')
    run(ad, VAL, f'_kveval{ck}')
print('VAL DONE')
import glob
best, bestf = None, -1.0
for ck in CKPTS:
    d = json.load(open(f'{results}/scores_smolvlm2-2.2b_kv_kveval{ck}.json'))
    print(f'checkpoint-{ck}: val_f1={d["field_f1"]:.4f}')
    if d['field_f1'] > bestf:
        best, bestf = ck, d['field_f1']
print(f'WINNER: checkpoint-{best} ({bestf:.4f})')
run(get_adapter(f'checkpoint-{best}'), TEST, '_kvtest')
from huggingface_hub import HfApi
api = HfApi(token=os.environ.get('HF_TOKEN')) if os.environ.get('HF_TOKEN') else None
if api is None:
    print('skipping Hub sync (no token); download results via: kaggle kernels output <slug>')
if api is not None:
    for f in glob.glob(f'{results}/scores_smolvlm2-2.2b_kv_*.json') + glob.glob(f'{results}/preds_smolvlm2-2.2b_kv_*.jsonl'):
        api.upload_file(path_or_fileobj=f, path_in_repo='eval/' + f.split('/')[-1],
                        repo_id=HUB_RES, repo_type='model')
    print('results synced to hub')
print('results synced to hub')
# v6: public clean-eval dataset (revert to private post-eval)
